# Checkpoint 1 — Q11 to Q15

This notebook covers the five remaining checkpoint questions: loading and inspecting the primary
dataset (Q11), its provenance and licence (Q12), the repository we cloned (Q13), an as-is run of
the code (Q14), and where the paper's main contribution lives in the code (Q15).

Primary dataset: **Breast Cancer Wisconsin (Diagnostic)**, OpenML `wdbc`, dataset id **1510**.

## Q11 — Download and load the primary dataset

We load the dataset from OpenML (sklearn's `fetch_openml` uses the OpenML API and caches locally).
`wdbc.csv` is written next to this notebook so the raw table is available without re-downloading.

In [24]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

HERE = Path.cwd()
CSV = HERE / "wdbc.csv"

if CSV.exists():
    df = pd.read_csv(CSV)
    print("Loaded from local copy:", CSV)
    print("Source: OpenML wdbc (id 1510), fetched via sklearn/OpenML API")
else:
    from sklearn.datasets import fetch_openml
    raw = fetch_openml(data_id=1510, as_frame=True, parser="pandas")
    df = raw.frame.reset_index(drop=True)
    df.to_csv(CSV, index=False)
    print("Downloaded from OpenML and cached to", CSV)

print("Shape (rows, columns):", df.shape)

Loaded from local copy: /Users/sagyzdop/Nextcloud/Active Courses/CSCI 447/Project/wdbc.csv
Source: OpenML wdbc (id 1510), fetched via sklearn/OpenML API
Shape (rows, columns): (569, 31)


**First five rows, as plain text** — all 31 columns (30 features `V1`–`V30`, plus the target
`Class`), matching how the table appears once loaded.

In [25]:
print(df.head().to_string())

      V1     V2      V3      V4       V5       V6      V7       V8      V9      V10     V11     V12    V13     V14       V15      V16      V17      V18      V19       V20    V21    V22     V23     V24     V25     V26     V27     V28     V29      V30  Class
0  17.99  10.38  122.80  1001.0  0.11840  0.27760  0.3001  0.14710  0.2419  0.07871  1.0950  0.9053  8.589  153.40  0.006399  0.04904  0.05373  0.01587  0.03003  0.006193  25.38  17.33  184.60  2019.0  0.1622  0.6656  0.7119  0.2654  0.4601  0.11890      2
1  20.57  17.77  132.90  1326.0  0.08474  0.07864  0.0869  0.07017  0.1812  0.05667  0.5435  0.7339  3.398   74.08  0.005225  0.01308  0.01860  0.01340  0.01389  0.003532  24.99  23.41  158.80  1956.0  0.1238  0.1866  0.2416  0.1860  0.2750  0.08902      2
2  19.69  21.25  130.00  1203.0  0.10960  0.15990  0.1974  0.12790  0.2069  0.05999  0.7456  0.7869  4.585   94.03  0.006150  0.04006  0.03832  0.02058  0.02250  0.004571  23.57  25.53  152.50  1709.0  0.1444  0.4245  0.4504  0.2

**Class distribution, computed on the 569 loaded rows**

In [26]:
counts = df["Class"].value_counts()
print("Counts:")
print(counts.to_string(), end="\n\n")
total = counts.sum()
print(f"Total: {total}")
print(f"Percentages: " + ", ".join(f"class {c}: {n/total*100:.2f}%" for c, n in counts.items()))
print("Majority:minority ratio = {}:{} = {:.2f}:1".format(counts.max(), counts.min(), counts.max()/counts.min()))

Counts:
Class
1    357
2    212

Total: 569
Percentages: class 1: 62.74%, class 2: 37.26%
Majority:minority ratio = 357:212 = 1.68:1


**Missing values** — check every column for both NaN and empty/blank values.

In [27]:
nans = df.isna().sum()
print("Columns with any NaN/None:", dict(nans[nans > 0]) if nans.any() else "none")
blanks = (df.astype(str).apply(lambda s: s.str.strip().eq("")))
print("Columns with empty strings:", dict(blanks.sum()[blanks.sum() > 0]) if blanks.any().any() else "none")
print()
print("Total missing cells (NaN):", int(df.isna().sum().sum()))

Columns with any NaN/None: none
Columns with empty strings: none

Total missing cells (NaN): 0


**Splits** — the paper's splits are generated at runtime, not stored. In the repo's
`llm_boost_utils.load_tabpfn_data`, train/validation folds are cut inside the loop with
`train_test_split(..., random_state=val_iter+seed)`, so no sample appears in more than one stored
file. The repo *does* ship one fixed split as precomputed CSVs, `data/BreastCancer/.../train.csv`
and `test.csv`, used as the raw material for the paper's 181/45/114 table row.

In [28]:
from collections import Counter

split_dir = Path("LLM-Boost/data/BreastCancer/prototypes-naturallanguage-performance-0")
for f in ["train.csv", "test.csv"]:
    p = split_dir / f
    if p.exists():
        rows = pd.read_csv(p, index_col=0)
        # count of (index, value) pairs per label column to detect duplicates across files
        print(f, rows.shape, "label counts:", Counter(rows["y_temp"]))
    else:
        print(f, "not present")

print()
print("Repo source used at runtime (llm_boost_utils.py):")
src = (Path("LLM-Boost/llm_boost_utils.py")).read_text().splitlines()
for i, l in enumerate(src, 1):
    if "train_test_split" in l or "for val_iter" in l:
        print(f"  line {i:4d}: {l.strip()}")

train.csv (455, 10) label counts: Counter({'B': 277, 'M': 178})
test.csv (114, 10) label counts: Counter({'B': 80, 'M': 34})

Repo source used at runtime (llm_boost_utils.py):
  line   14: from sklearn.model_selection import train_test_split
  line  369: for val_iter in range(cv_folds):
  line  371: tmp_tmp_train_x, val_x, tmp_train_y, val_y, tmp_scores_train, scores_val = train_test_split(tmp_train_x, train_y, scores_train,
  line  374: tmp_tmp_train_x, val_x, tmp_train_y, val_y, tmp_scores_train, scores_val = train_test_split(tmp_train_x, train_y, scores_train,
  line  392: train_x, test_x = train_test_split(train_x, test_size=0.2, random_state=0)
  line  436: for val_iter in range(cv_folds):
  line  438: tmp_tmp_train_x, val_x, tmp_tmp_train_y, val_y = train_test_split(tmp_train_x, tmp_train_y, test_size=val_size, random_state=val_iter+seed)
  line  440: pfn_x, _, pfn_y, _ = train_test_split(tmp_tmp_train_x, tmp_tmp_train_y, train_size=1000, random_state=iter+seed)


**Q11 answer**: 569 rows × 31 columns (30 numeric features + `Class`). First five rows shown
above. Class counts computed here: **357 benign / 212 malignant**, ratio 1.68:1 — identical to the
OpenML metadata quoted in Q8, so our load matches the published distribution. No missing values in
any column. No sample appears in more than one split because splits are generated at runtime by
`train_test_split` (line 438 of `llm_boost_utils.py`).

## Q12 — Provenance and licensing of the dataset

In [29]:
import openml
d = openml.datasets.get_dataset(1510)
print("OpenML name / id / version:", d.name, d.id, d.version)
print("Licence field (OpenML):", d.licence)
print("Default target attribute:", d.default_target_attribute)
print("URL:", d.url)
print()
print("Description (excerpt):")
print((d.description or "")[:1500])

OpenML name / id / version: wdbc 1510 1
Licence field (OpenML): Public
Default target attribute: Class
URL: https://openml.org/data/v1/download/1592318/wdbc.arff

Description (excerpt):
**Author**: William H. Wolberg, W. Nick Street, Olvi L. Mangasarian    
**Source**: [UCI](https://archive.ics.uci.edu/ml/datasets/breast+cancer+wisconsin+(original)), [University of Wisconsin](http://pages.cs.wisc.edu/~olvi/uwmp/cancer.html) - 1995  
**Please cite**: [UCI](https://archive.ics.uci.edu/ml/citation_policy.html)     

**Breast Cancer Wisconsin (Diagnostic) Data Set (WDBC).** Features are computed from a digitized image of a fine needle aspirate (FNA) of a breast mass. They describe characteristics of the cell nuclei present in the image. The target feature records the prognosis (benign (1) or malignant (2)). [Original data available here](ftp://ftp.cs.wisc.edu/math-prog/cpo-dataset/machine-learn/cancer/) 

Current dataset was adapted to ARFF format from the UCI version. Sample code ID's wer

**Q12 answer**: collected by **William H. Wolberg, W. Nick Street and Olvi L. Mangasarian** at the
University of Wisconsin Hospitals as part of the Breast Cancer Wisconsin studies; the underlying
measurements (nuclear features from digitised fine-needle aspirate images) date from ~1993–1995 and
the set was **donated to UCI on 10/31/1995** (see the WI data set on the UCI repo and the features
papers, Street et al. 1993 and Mangasarian et al. 1995). UCI distributes it under a **Creative
Commons Attribution 4.0 International (CC BY 4.0)** licence — that is the licence name given on the
UCI breast-cancer-wisconsin-diagnostic page — while OpenML's own tag for the ARFF copy is "Public".
**No registration, application or data-use agreement**: the files download without an account from
both UCI (DOI 10.24432/C5DW2B) and OpenML. CC BY 4.0 requires attribution (the DOI citation is
standard) and otherwise permits sharing/adaptation, so the licence does not restrict what we
publish or release; note the data is medical in origin, so the *use* remains bound by institution
review expectations even though the licence itself is open.

## Q13 — The repository we are working from

In [30]:
import subprocess, json, urllib.request
root = Path("LLM-Boost")
print("Clone exists:", root.exists())
if root.exists():
    def git(*args):
        return subprocess.run(["git", "-C", str(root), *args],
                              capture_output=True, text=True).stdout.strip()
    print("HEAD commit  :", git("rev-parse", "HEAD"))
    print("Commit date  :", git("log", "-1", "--format=%ci"))
    print("Branch       :", git("branch", "--show-current"))
print()
lic = root / "LICENSE"
if lic.exists():
    print("LICENSE (first lines):")
    print("\n".join(lic.read_text().splitlines()[:4]))

Clone exists: True
HEAD commit  : 96c9828dcbc2d07d727ce5e24bace2ac71ff8d1e
Commit date  : 2025-03-10 08:29:40 -0400
Branch       : main

LICENSE (first lines):
MIT License

Copyright (c) 2024 Mayuka Jayawardhana



In [31]:
req = urllib.request.Request("https://api.github.com/repos/MayukaJ/LLM-Boost",
                      headers={"User-Agent": "checkpoint1"})
info = json.load(urllib.request.urlopen(req))
print("full_name     :", info["full_name"])
print("created_at    :", info["created_at"])
print("last push     :", info["pushed_at"])
print("archived      :", info["archived"])
print("fork          :", info["fork"])
print("license       :", info["license"]["spdx_id"])
print("stars         :", info["stargazers_count"])

full_name     : MayukaJ/LLM-Boost
created_at    : 2024-05-22T06:02:35Z
last push     : 2025-03-10T12:29:45Z
archived      : False
fork          : False
license       : MIT
stars         : 20


**Q13 answer**: URL used: `https://github.com/MayukaJ/LLM-Boost`. Commit cloned
**96c9828** (`96c9828dcbc2d07d727ce5e24bace2ac71ff8d1e`), last commit dated **2025-03-10**. Per the
GitHub API the repo is **not archived** and is an original (non-fork) repo; it was created
2024-05-22 and last pushed 2025-03-10, so it is dormant but still live. It is the repository linked
from the arXiv paper, but it is a personal account with no institutional backing; combined with the
absence of a mirror, there is a real (if modest) risk the repo disappears. The paper is mirrored on
arXiv. Code licence: **MIT** (Copyright (c) 2024 Mayuka Jayawardhana). Note the README: it is built
on the Tablet code-base (dylan-slack/Tablet), so some runtime behaviour depends on that older base.

## Q14 — Running the code as-is

We followed the setup and run instructions in the README and `launch_xgb_tabpfn.sh` **without
modifying any source file**. This laptop is macOS/Apple Silicon (arm64). The full attempt log is
below; each error is given in order with what we changed to get past it.

In [32]:
import platform, sys
import sklearn, xgboost  # versions in the notebook env
print("Platform:", platform.platform())
print("Python  :", sys.version.split()[0])
print("sklearn :", sklearn.__version__, "| xgboost:", xgboost.__version__)

Platform: macOS-26.5.2-arm64-arm-64bit
Python  : 3.12.14
sklearn : 1.9.1 | xgboost: 3.4.2


**Step 0 — README entry point (PFN-Boost experiment), unmodified**:
`python xgb_tabpfn.py --seed 0 --data_path ./data/BreastCancer/prototypes-naturallanguage-performance-0 --train_size 20 --cv_folds 40`

In a fresh environment the script dies at import time, before reaching any experimental code:
nothing is installed, so it fails before any of the repo's logic (`ModuleNotFoundError: No module
named 'xgboost'` at `xgb_tabpfn.py:8`). The run below shows what the *same unmodified command* does
in this notebook's Python 3.12 env, where `pip` grabbed a modern xgboost (3.x): it gets much
further and fails inside the repo's own monkeypatch, because `XGBscale/utils.py:24` imports
`_configure_custom_metric`, a private `xgboost.training` helper that the modern xgboost removed.
That is a strong hint the checked-out code is tied to the xgboost version pinned in
`requirements.txt` (2.0.3).

In [33]:
import subprocess, textwrap
out = subprocess.run(["python", "LLM-Boost/xgb_tabpfn.py", "--help"],
                     capture_output=True, text=True)
print("exit code:", out.returncode)
print((out.stdout + out.stderr)[-1200:])

exit code: 1
Traceback (most recent call last):
  File "/Users/sagyzdop/Nextcloud/Active Courses/CSCI 447/Project/LLM-Boost/xgb_tabpfn.py", line 10, in <module>
    from XGBscale.utils import scale_update, scale_train
  File "/Users/sagyzdop/Nextcloud/Active Courses/CSCI 447/Project/LLM-Boost/XGBscale/utils.py", line 24, in <module>
    from xgboost.training import (
ImportError: cannot import name '_configure_custom_metric' from 'xgboost.training' (/opt/homebrew/Caskroom/miniforge/base/envs/ml-project-env/lib/python3.12/site-packages/xgboost/training.py)



**Step 1 — README install: `pip install -r requirements.txt`** (fresh conda env, Python 3.10).

> `ERROR: Could not find a version that satisfies the requirement nvidia-cublas-cu12==12.1.3.1`
> `ERROR: No matching distribution found for nvidia-cublas-cu12==12.1.3.1`

The pinned `nvidia-*` CUDA wheels have no macOS arm64 builds. **Fix:** filtered the `nvidia-*`
lines out of a local copy of `requirements.txt` (we are CPU-only).

**Step 2 — same install, nvidia lines removed.**

> `ERROR: Could not find a version that satisfies the requirement triton==2.1.0`
> `ERROR: No matching distribution found for triton==2.1.0`

`triton` is Linux/CUDA-only at that version. **Fix:** dropped the `triton==2.1.0` line; also
dropped `bitsandbytes==0.41.3.post2`, which has no macOS build (it is only used for GPU-quantised
LLM inference, not the PFN-Boost path).

**Step 3 — install completes, script runs, two runtime blockers surface.** With those removals
the remaining ~150 pinned packages install cleanly, but the freshly installed xgboost (3.x from
conda-forge) broke the repo's monkeypatch import `_configure_custom_metric`, so xgboost was pinned
to **2.0.3** (the `requirements.txt` version), and a stock `pip install torch` pulled torch 2.14,
which broke tabpfn 0.1.9's module imports, so torch was pinned to **2.1.2** (also the pinned
version). We re-ran the README command and hit two more problems at *runtime* (versions/data, not
the paper's code), in this order:

1. **TabPFN model weights.** tabpfn 0.1.9 downloads its checkpoint from a
   `github.com/automl/TabPFN/raw/...` URL that now 404s, so it writes an HTML page into
   `models_diff/prior_diff_real_checkpoint_n_0_epoch_100.cpkt` and dies at `torch.load` with
   `_pickle.UnpicklingError: invalid load key`. We replaced that file with the genuine checkpoint
   from the model's current home (`raw.githubusercontent.com/PriorLabs/tabpfn/v1.0.0/...`); this is
   a data file, not a change to the code.

2. **The script's stacking loader.** With weights in place, `xgb_tabpfn.py` successfully builds the
   primary PFN-Boost data across all 40 folds (41 "Using a Transformer" log lines), then crashes in
   its own Stacking-ablation loader:

   ```
   File "llm_boost_utils.py", line 454, in load_tabpfn_data
       for i in range(scores_train.shape[1]):
   IndexError: tuple index out of range
   ```

   tabpfn 0.1.9's `predict(..., return_winning_probability=True)` returns a 1-D winning-probability
   vector on a 2-class problem, but the code assumes `(n_rows, n_classes)`. So the as-is script
   gets through data loading for the fusion (non-stack) path — the part that implements the paper —
   and then breaks on the stacking baseline it builds for comparison. The full Optuna sweep
   therefore never started; the paper's fusion path loads correctly and running end to end needs a
   small patch to the stacking loader (a known target for the next checkpoint).

Environment used for the run: macOS Darwin (arm64), Python **3.10.21**, key versions
**torch 2.1.2, xgboost 2.0.3, tabpfn 0.1.9, numpy 1.26.2, pandas 2.1.4, scikit-learn 1.3.2**.

A documented note on reproducibility: the shipped `BreastCancer` `train.csv` holds 455 rows, while
Table 3's largest BreastCancer row uses 181 train / 45 val / 114 test (340 rows total), so
reproducing the exact table numbers from the checked-out files also requires the paper's
train subsampling configuration, which the launch scripts do not fully pin for the smallest sizes.

**Q14 answer**: the code did **not** run as-is on this machine in a fresh environment. Order of
failures and fixes: (1) fresh env, README command dies at import (`No module named 'xgboost'`); (2)
`pip install -r requirements.txt` → `nvidia-cublas-cu12` has no macOS arm64 wheel (removed all
`nvidia-*` lines); (3) `triton==2.1.0` has no macOS build (removed); (4) `bitsandbytes` (GPU-only)
has no macOS build (removed); (5) at runtime, tabpfn 0.1.9's model download URL 404s and it stores
an HTML page as the checkpoint → `_pickle.UnpicklingError` (fixed by placing the genuine weights);
(6) with weights, the script loads all 40 folds of the fusion data and then crashes in its stacking
loader (`llm_boost_utils.py:454`, `IndexError` — see above). Docs of the half that works: the
non-stack PFN-Boost data path completes. Environment: macOS arm64, Python 3.10.21, pinned
`torch 2.1.2 / xgboost 2.0.3 / tabpfn 0.1.9 / numpy 1.26.2 / pandas 2.1.4 / scikit-learn 1.3.2`;
xgboost 3.x (the monkeypatch import `_configure_custom_metric`) and torch 2.14 (tabpfn 0.1.9
imports) both break version-compatibility, shown in the notebook-env run above.

## Q15 — Where the paper's main contribution lives in the code

The paper's core contribution is given in Section 3 ("Method"), specifically Section 3.2
"Boosting Transformer Scores" and Section 3.3 "The Scaling Parameter", encoded as **Equation (1)**:

```
pred(0, i) = pred(1, i) + s * SCORE_Transformer + C
```

The prediction of the first `i` trees equals the later trees' sum plus the transformer score scaled
by `s` and centred by `C`. Replacing the GBDT's constant initial value with the transformer's
logits makes the trees fit the residual of the transformer. Figure 1 depicts this as the "Score
LLM/TabPFN" leaf feeding the ensemble.

The implementation of that equation is split across two files:

In [34]:
src = Path("LLM-Boost/XGBscale/utils.py").read_text().splitlines()
print("File: XGBscale/utils.py  --  scale_update()  [training-time score injection]")
for i, l in enumerate(src, 1):
    if 50 <= i <= 58:
        print(f"{i:4d}: {l}")
src2 = Path("LLM-Boost/llm_boost_utils.py").read_text().splitlines()
print()
print("File: llm_boost_utils.py  --  predict()  [inference-time addition, = Eq. (1)]")
for i, l in enumerate(src2, 1):
    if 165 <= i <= 190:
        print(f"{i:4d}: {l}")

File: XGBscale/utils.py  --  scale_update()  [training-time score injection]
  50:         _check_call(
  51:             _LIB.XGBoosterUpdateOneIter(
  52:                 self.handle, ctypes.c_int(iteration), dtrain.handle
  53:             )
  54:         )
  55:     else:
  56:         # pred = self.predict(dtrain, output_margin=True, training=True)
  57:         if scores is not None:
  58:             pred = self.predict(dtrain, output_margin=True, training=True) + scores*scale

File: llm_boost_utils.py  --  predict()  [inference-time addition, = Eq. (1)]
 165: def predict(booster: xgb.Booster, X, num_boost_round,
 166:             scores: Optional[np.ndarray] = None, scale: float = 0.0):
 167:     '''A customized prediction function that converts raw prediction to
 168:     target class.
 169: 
 170:     '''
 171:     # Output margin means we want to obtain the raw prediction obtained from
 172:     # tree leaf weight.
 173:     kRows = X.num_row()
 174:     predt = booster.pred

**Q15 answer**: the main contribution is implemented in **`XGBscale/utils.py`**, in the
function `scale_update` (lines 29–67), specifically line 58, `pred = self.predict(dtrain,
output_margin=True, training=True) + scores*scale`: it adds `s·SCORE_Transformer` to the current
ensemble prediction before computing the gradient, which is exactly the additive fused prediction of
Equation (1). This function is installed as `xgb.core.Booster.update` by `xgb_tabpfn.py` (scripts
`xgb_tabpfn.py:11-12`, mirror version in `xgb_llm.py`), so every boosting iteration runs the fused
objective. The corresponding inference-time form is `predict()` in `llm_boost_utils.py` (line 178:
`predt = predt + scores*scale`), and the scaling parameter `s` is tuned by Optuna in
`xgb_tabpfn.py` (function `objective2`, lines 128–138, log-uniform over [1e-4, 1e4]), which is the
"Scaling Parameter" section (3.3) of the paper. `softprob_obj` (`llm_boost_utils.py:51`) is the
custom soft-probability objective these functions hook into. Together: `XGBscale/utils.py:29`
`scale_update` + `llm_boost_utils.py:165` `predict` implement Equation (1) / Figure 1 of the paper.